In [1]:
#!pip install -qqq arize-otel

In [2]:
import os
space_id = os.environ["ARIZE_SPACE_ID"]
api_key = os.environ["ARIZE_API_KEY"]

In [3]:
project_name = "ae_triage_agent"

In [4]:
from arize.otel import register
from opentelemetry import trace
from opentelemetry.trace.status import Status, StatusCode
import json
import time
import random

tracer_provider = register(
    space_id=space_id,
    api_key=api_key,
    project_name=project_name,
)
tracer = trace.get_tracer(__name__)

🔭 OpenTelemetry Tracing Details 🔭
|  Arize Project: ae_triage_agent
|  Span Processor: BatchSpanProcessor
|  Collector Endpoint: otlp.arize.com
|  Transport: gRPC
|  Transport Headers: {'authorization': '****', 'api_key': '****', 'arize-space-id': '****', 'space_id': '****', 'arize-interface': '****'}
|  
|  Using a default SpanProcessor. `add_span_processor` will overwrite this default.
|  
|  `register` has set this TracerProvider as the global OpenTelemetry default.
|  To disable this behavior, call `register` with `set_global_tracer_provider=False`.



In [5]:
# Synthetic data for AE Triage Agent (Step 1)
# Fake AE report, MedDRA corpus, SOP corpus, and expected classification.

AE_REPORT_RAW = (
    "62-year-old female receiving darzumab for DLBCL reported Grade 3 cytokine release syndrome "
    "on Day 2 of Cycle 1. Patient hospitalized, treated with tocilizumab. Symptoms resolved within 48 hours."
)

MEDDRA_CORPUS = [
    {"document_id": "meddra_10001234", "preferred_term": "Cytokine release syndrome", "soc_code": "10021428", "soc_name": "Immune system disorders", "content": "Cytokine release syndrome: systemic inflammatory response following immune activation (e.g. CAR-T, bispecific antibodies). May include fever, hypotension, hypoxia. Grading per ASTCT."},
    {"document_id": "meddra_10023456", "preferred_term": "Infusion related reaction", "soc_code": "10021428", "soc_name": "Immune system disorders", "content": "Infusion related reaction: signs/symptoms during or shortly after IV infusion. Includes fever, chills, flushing, dyspnea, hypotension."},
    {"document_id": "meddra_10034567", "preferred_term": "Immune effector cell-associated neurotoxicity syndrome", "soc_code": "10029205", "soc_name": "Nervous system disorders", "content": "Immune effector cell-associated neurotoxicity syndrome (ICANS): neurotoxicity after immune effector cell therapy. Encephalopathy, aphasia, seizures, cerebral edema."},
    {"document_id": "meddra_10045678", "preferred_term": "Febrile neutropenia", "soc_code": "10029104", "soc_name": "Blood and lymphatic system disorders", "content": "Febrile neutropenia: fever with neutrophil count below 0.5 x 10^9/L. Common in chemotherapy."},
    {"document_id": "meddra_10056789", "preferred_term": "Tumour lysis syndrome", "soc_code": "10027433", "soc_name": "Metabolism and nutrition disorders", "content": "Tumour lysis syndrome: metabolic derangements from rapid tumour cell death. Hyperuricaemia, hyperkalaemia, hyperphosphataemia, hypocalcaemia."},
]

# Top 3 for retrieval: exact match ~0.94, near-misses ~0.78, ~0.71
MEDDRA_TOP_3 = [
    {"document_id": MEDDRA_CORPUS[0]["document_id"], "preferred_term": MEDDRA_CORPUS[0]["preferred_term"], "soc_code": MEDDRA_CORPUS[0]["soc_code"], "soc_name": MEDDRA_CORPUS[0]["soc_name"], "content": MEDDRA_CORPUS[0]["content"], "score": 0.94},
    {"document_id": MEDDRA_CORPUS[1]["document_id"], "preferred_term": MEDDRA_CORPUS[1]["preferred_term"], "soc_code": MEDDRA_CORPUS[1]["soc_code"], "soc_name": MEDDRA_CORPUS[1]["soc_name"], "content": MEDDRA_CORPUS[1]["content"], "score": 0.78},
    {"document_id": MEDDRA_CORPUS[2]["document_id"], "preferred_term": MEDDRA_CORPUS[2]["preferred_term"], "soc_code": MEDDRA_CORPUS[2]["soc_code"], "soc_name": MEDDRA_CORPUS[2]["soc_name"], "content": MEDDRA_CORPUS[2]["content"], "score": 0.71},
]

SOP_CORPUS = [
    {"document_id": "SOP-PV-003-v2.1-§4.2", "sop_number": "SOP-PV-003", "version": "2.1", "section": "4.2", "content": "Seriousness classification: hospitalization, life-threatening, death, disability, congenital anomaly, or other medically important condition. Serious cases require human review before submission; auto-submit is not permitted."},
    {"document_id": "SOP-PV-004-v1.2-§3.1", "sop_number": "SOP-PV-004", "version": "1.2", "section": "3.1", "content": "Expectedness: compare to reference safety information (RSI) and product label. Darzumab label lists cytokine release syndrome as expected; Grade 3 CRS listed in Warnings."},
    {"document_id": "SOP-PV-005-v1.0-§2", "sop_number": "SOP-PV-005", "version": "1.0", "section": "2", "content": "Auto-submission policy: only non-serious, expected, and not reasonably related cases may be auto-submitted. All serious cases require PV analyst review and sign-off."},
]

SOP_TOP_3 = [
    {"document_id": SOP_CORPUS[0]["document_id"], "content": SOP_CORPUS[0]["content"], "score": 0.91},
    {"document_id": SOP_CORPUS[1]["document_id"], "content": SOP_CORPUS[1]["content"], "score": 0.87},
    {"document_id": SOP_CORPUS[2]["document_id"], "content": SOP_CORPUS[2]["content"], "score": 0.82},
]

EXPECTED_CLASSIFICATION = {
    "seriousness": "serious",
    "seriousness_criteria": ["hospitalization"],
    "expectedness": "expected",
    "causality": "possibly_related",
    "meddra_pt": "Cytokine release syndrome",
    "meddra_soc": "Immune system disorders",
}

PARSED_INTAKE = {
    "patient_age": 62,
    "patient_sex": "female",
    "drug": "darzumab",
    "indication": "DLBCL",
    "event_description": "Grade 3 cytokine release syndrome",
    "onset": "Day 2 of Cycle 1",
    "outcome": "Symptoms resolved within 48 hours",
    "hospitalization": True,
    "treatment": "tocilizumab",
}

In [6]:
def build_trace_scenario_a(case_id: str, session_id: str):
    """Scenario A: read-only trace. Root + Spans 1-4 only. Root status OK."""
    chain_name = "AE Triage Agent"
    latency_ms = random.uniform(50, 150)
    time.sleep(0.002)

    with tracer.start_as_current_span(chain_name) as chain_span:
        chain_span.set_attribute("openinference.span.kind", "CHAIN")
        chain_span.set_attribute("input.value", AE_REPORT_RAW)
        chain_span.set_attribute("input.mime_type", "text/plain")
        chain_span.set_attribute("session.id", session_id)
        chain_span.set_attribute("user.id", "pv_analyst_001")
        chain_span.set_attribute("metadata.case_id", case_id)

        # Agent 1: Report Intake Agent (parent of AE Report Intake)
        with tracer.start_as_current_span("Report Intake Agent") as agent1_span:
            agent1_span.set_attribute("openinference.span.kind", "AGENT")
            agent1_span.set_attribute("agent.name", "report_intake_agent")
            agent1_span.set_attribute("input.value", AE_REPORT_RAW)
            agent1_span.set_attribute("input.mime_type", "text/plain")

            # Child 1: AE Report Intake (CHAIN), with two retriever children
            with tracer.start_as_current_span("AE Report Intake") as span1:
                span1.set_attribute("openinference.span.kind", "CHAIN")
                span1.set_attribute("input.value", AE_REPORT_RAW)
                span1.set_attribute("output.value", json.dumps(PARSED_INTAKE))
                span1.set_attribute("output.mime_type", "application/json")
                span1.set_status(Status(StatusCode.OK))

                # Child 2: MedDRA Term Lookup (under AE Report Intake)
                with tracer.start_as_current_span("MedDRA Term Lookup") as span2:
                    span2.set_attribute("openinference.span.kind", "RETRIEVER")
                    span2.set_attribute("input.value", "cytokine release syndrome grade 3")
                    span2.set_attribute("output.value", json.dumps([{"document_id": d["document_id"], "score": d["score"]} for d in MEDDRA_TOP_3]))
                    span2.set_attribute("output.mime_type", "application/json")
                    for idx, doc in enumerate(MEDDRA_TOP_3):
                        span2.set_attribute(f"retrieval.documents.{idx}.document.id", doc["document_id"])
                        span2.set_attribute(f"retrieval.documents.{idx}.document.content", doc["content"])
                        span2.set_attribute(f"retrieval.documents.{idx}.document.score", doc["score"])
                    span2.set_status(Status(StatusCode.OK))

                # Child 3: SOP & Product Label Retrieval (under AE Report Intake)
                with tracer.start_as_current_span("SOP & Product Label Retrieval") as span3:
                    span3.set_attribute("openinference.span.kind", "RETRIEVER")
                    span3.set_attribute("input.value", "darzumab cytokine release syndrome seriousness classification")
                    span3.set_attribute("output.value", json.dumps([{"document_id": d["document_id"], "score": d["score"]} for d in SOP_TOP_3]))
                    span3.set_attribute("output.mime_type", "application/json")
                    for idx, doc in enumerate(SOP_TOP_3):
                        span3.set_attribute(f"retrieval.documents.{idx}.document.id", doc["document_id"])
                        span3.set_attribute(f"retrieval.documents.{idx}.document.content", doc["content"])
                        span3.set_attribute(f"retrieval.documents.{idx}.document.score", doc["score"])
                    span3.set_status(Status(StatusCode.OK))

            agent1_span.set_attribute("output.value", json.dumps(PARSED_INTAKE))
            agent1_span.set_attribute("output.mime_type", "application/json")

        # Agent 2: Triage Classification Agent (parent of Classification Reasoning)
        with tracer.start_as_current_span("Triage Classification Agent") as agent2_span:
            agent2_span.set_attribute("openinference.span.kind", "AGENT")
            agent2_span.set_attribute("agent.name", "triage_classification_agent")
            agent2_span.set_attribute("input.value", AE_REPORT_RAW[:500])
            agent2_span.set_attribute("input.mime_type", "text/plain")

            # Child 4: Classification Reasoning (LLM)
            with tracer.start_as_current_span("Classification Reasoning") as span4:
                span4.set_attribute("openinference.span.kind", "LLM")
                span4.set_attribute("llm.model_name", "gpt-4o")
                span4.set_attribute("llm.provider", "openai")
                span4.set_attribute("llm.invocation_parameters", json.dumps({"temperature": 0.0}))
                span4.set_attribute("llm.input_messages.0.message.role", "system")
                span4.set_attribute("llm.input_messages.0.message.content", "You are a PV triage assistant. Classify the AE using MedDRA and SOP context.")
                span4.set_attribute("llm.input_messages.1.message.role", "user")
                span4.set_attribute("llm.input_messages.1.message.content", f"Report: {AE_REPORT_RAW[:500]}... [Retrieved MedDRA and SOP chunks omitted for brevity]")
                span4.set_attribute("llm.output_messages.0.message.role", "assistant")
                span4.set_attribute("llm.output_messages.0.message.content", json.dumps(EXPECTED_CLASSIFICATION))
                span4.set_attribute("llm.token_count.prompt", 1800)
                span4.set_attribute("llm.token_count.completion", 120)
                span4.set_attribute("llm.token_count.total", 1920)
                span4.set_attribute("input.value", AE_REPORT_RAW[:500])
                span4.set_attribute("output.value", json.dumps(EXPECTED_CLASSIFICATION))
                span4.set_attribute("llm.prompt_template.template", "You are a PV triage assistant. Classify the AE using MedDRA and SOP context.\n\nReport: {report_text}")
                span4.set_attribute("llm.prompt_template.version", "v1")
                span4.set_attribute("llm.prompt_template.variables", json.dumps({"report_text": AE_REPORT_RAW[:500] + "..."}))
                span4.set_status(Status(StatusCode.OK))

            agent2_span.set_attribute("output.value", json.dumps(EXPECTED_CLASSIFICATION))
            agent2_span.set_attribute("output.mime_type", "application/json")

        classification_summary = json.dumps(EXPECTED_CLASSIFICATION)
        chain_span.set_attribute("output.value", classification_summary[:500] if len(classification_summary) > 500 else classification_summary)
        chain_span.set_attribute("output.mime_type", "application/json")
        chain_span.set_attribute("latency_ms", round(latency_ms, 2))
        chain_span.set_status(Status(StatusCode.OK))

In [7]:
def build_trace_scenario_b(case_id: str, session_id: str):
    """Scenario B: blocked tool + cascade. Root + Spans 1-7. Root status ERROR."""
    chain_name = "AE Triage Agent"
    latency_ms = random.uniform(80, 200)
    time.sleep(0.002)

    with tracer.start_as_current_span(chain_name) as chain_span:
        chain_span.set_attribute("openinference.span.kind", "CHAIN")
        chain_span.set_attribute("input.value", AE_REPORT_RAW)
        chain_span.set_attribute("input.mime_type", "text/plain")
        chain_span.set_attribute("session.id", session_id)
        chain_span.set_attribute("user.id", "pv_analyst_001")
        chain_span.set_attribute("metadata.case_id", case_id)

        # Agent 1: Report Intake Agent (parent of AE Report Intake)
        with tracer.start_as_current_span("Report Intake Agent") as agent1_span:
            agent1_span.set_attribute("openinference.span.kind", "AGENT")
            agent1_span.set_attribute("agent.name", "report_intake_agent")
            agent1_span.set_attribute("input.value", AE_REPORT_RAW)
            agent1_span.set_attribute("input.mime_type", "text/plain")

            # Child 1: AE Report Intake (CHAIN), with two retriever children
            with tracer.start_as_current_span("AE Report Intake") as span1:
                span1.set_attribute("openinference.span.kind", "CHAIN")
                span1.set_attribute("input.value", AE_REPORT_RAW)
                span1.set_attribute("output.value", json.dumps(PARSED_INTAKE))
                span1.set_attribute("output.mime_type", "application/json")
                span1.set_status(Status(StatusCode.OK))

                # Child 2: MedDRA Term Lookup (under AE Report Intake)
                with tracer.start_as_current_span("MedDRA Term Lookup") as span2:
                    span2.set_attribute("openinference.span.kind", "RETRIEVER")
                    span2.set_attribute("input.value", "cytokine release syndrome grade 3")
                    span2.set_attribute("output.value", json.dumps([{"document_id": d["document_id"], "score": d["score"]} for d in MEDDRA_TOP_3]))
                    span2.set_attribute("output.mime_type", "application/json")
                    for idx, doc in enumerate(MEDDRA_TOP_3):
                        span2.set_attribute(f"retrieval.documents.{idx}.document.id", doc["document_id"])
                        span2.set_attribute(f"retrieval.documents.{idx}.document.content", doc["content"])
                        span2.set_attribute(f"retrieval.documents.{idx}.document.score", doc["score"])
                    span2.set_status(Status(StatusCode.OK))

                # Child 3: SOP & Product Label Retrieval (under AE Report Intake)
                with tracer.start_as_current_span("SOP & Product Label Retrieval") as span3:
                    span3.set_attribute("openinference.span.kind", "RETRIEVER")
                    span3.set_attribute("input.value", "darzumab cytokine release syndrome seriousness classification")
                    span3.set_attribute("output.value", json.dumps([{"document_id": d["document_id"], "score": d["score"]} for d in SOP_TOP_3]))
                    span3.set_attribute("output.mime_type", "application/json")
                    for idx, doc in enumerate(SOP_TOP_3):
                        span3.set_attribute(f"retrieval.documents.{idx}.document.id", doc["document_id"])
                        span3.set_attribute(f"retrieval.documents.{idx}.document.content", doc["content"])
                        span3.set_attribute(f"retrieval.documents.{idx}.document.score", doc["score"])
                    span3.set_status(Status(StatusCode.OK))

            agent1_span.set_attribute("output.value", json.dumps(PARSED_INTAKE))
            agent1_span.set_attribute("output.mime_type", "application/json")

        # Agent 2: Triage Classification Agent (parent of Classification Reasoning, Tool, Recovery)
        with tracer.start_as_current_span("Triage Classification Agent") as agent2_span:
            agent2_span.set_attribute("openinference.span.kind", "AGENT")
            agent2_span.set_attribute("agent.name", "triage_classification_agent")
            agent2_span.set_attribute("input.value", AE_REPORT_RAW[:500])
            agent2_span.set_attribute("input.mime_type", "text/plain")

            # Child 4: Classification Reasoning (LLM)
            with tracer.start_as_current_span("Classification Reasoning") as span4:
                span4.set_attribute("openinference.span.kind", "LLM")
                span4.set_attribute("llm.model_name", "gpt-4o")
                span4.set_attribute("llm.provider", "openai")
                span4.set_attribute("llm.invocation_parameters", json.dumps({"temperature": 0.0}))
                span4.set_attribute("llm.input_messages.0.message.role", "system")
                span4.set_attribute("llm.input_messages.0.message.content", "You are a PV triage assistant. Classify the AE using MedDRA and SOP context.")
                span4.set_attribute("llm.input_messages.1.message.role", "user")
                span4.set_attribute("llm.input_messages.1.message.content", f"Report: {AE_REPORT_RAW[:500]}...")
                span4.set_attribute("llm.output_messages.0.message.role", "assistant")
                span4.set_attribute("llm.output_messages.0.message.content", json.dumps(EXPECTED_CLASSIFICATION))
                span4.set_attribute("llm.token_count.prompt", 1800)
                span4.set_attribute("llm.token_count.completion", 120)
                span4.set_attribute("llm.token_count.total", 1920)
                span4.set_attribute("input.value", AE_REPORT_RAW[:500])
                span4.set_attribute("output.value", json.dumps(EXPECTED_CLASSIFICATION))
                span4.set_attribute("llm.prompt_template.template", "You are a PV triage assistant. Classify the AE using MedDRA and SOP context.\n\nReport: {report_text}")
                span4.set_attribute("llm.prompt_template.version", "v1")
                span4.set_attribute("llm.prompt_template.variables", json.dumps({"report_text": AE_REPORT_RAW[:500] + "..."}))
                span4.set_status(Status(StatusCode.OK))

            # Child 5: TOOL — Submit to Safety Database (BLOCKED), nested under Classification Reasoning
            tool_params = {"case_id": case_id, "classification": EXPECTED_CLASSIFICATION, "auto_submit": True}
            exc_msg = "Write action blocked: serious_case_requires_human_review (SOP-PV-003 §4.2)"
            stacktrace_fake = "  File \"control_plane.py\", line 142\\n    raise PolicyEnforcementError(msg)\\nPolicyEnforcementError: Write action blocked..."
            recovery_output = '{"summary": "Tool was blocked; case requires human review.", "next_step": "escalate"}'
            with tracer.start_as_current_span("Submit to Safety Database") as span5:
                span5.set_attribute("openinference.span.kind", "TOOL")
                span5.set_attribute("tool.name", "safety_db_write")
                span5.set_attribute("tool.parameters", json.dumps(tool_params))
                span5.set_attribute("input.value", json.dumps(tool_params))
                span5.set_attribute("output.value", "")
                span5.add_event("exception", attributes={
                    "exception.type": "PolicyEnforcementError",
                    "exception.message": exc_msg,
                    "exception.stacktrace": stacktrace_fake,
                })
                span5.add_event("policy.action_blocked", attributes={
                    "rule": "serious_case_requires_human_review",
                    "enforced_by": "SOP-PV-003 §4.2",
                    "case_seriousness": "serious",
                    "required_action": "human_review_before_submission",
                })
                span5.set_status(Status(StatusCode.ERROR, exc_msg))

                # Child 6: LLM Recovery Reasoning (ERROR — malformed output), nested under Tool
                validation_err = "Agent output failed schema validation — missing required field 'disposition'"
                with tracer.start_as_current_span("Recovery Reasoning") as span6:
                    span6.set_attribute("openinference.span.kind", "LLM")
                    span6.set_attribute("llm.model_name", "gpt-4o")
                    span6.set_attribute("llm.provider", "openai")
                    span6.set_attribute("llm.input_messages.0.message.role", "user")
                    span6.set_attribute("llm.input_messages.0.message.content", "The safety_db_write was blocked. Provide a valid triage disposition (disposition field required).")
                    span6.set_attribute("llm.output_messages.0.message.role", "assistant")
                    span6.set_attribute("llm.output_messages.0.message.content", recovery_output)
                    span6.set_attribute("input.value", "The safety_db_write was blocked...")
                    span6.set_attribute("output.value", recovery_output)
                    span6.add_event("exception", attributes={
                        "exception.type": "OutputValidationError",
                        "exception.message": validation_err,
                    })
                    span6.set_status(Status(StatusCode.ERROR, validation_err))

            agent2_span.set_attribute("output.value", "Agent failed to produce valid triage output after tool rejection")
            agent2_span.set_attribute("output.mime_type", "text/plain")

        # Child 7: Final Output (ERROR), direct child of root — not nested
        final_err = "Agent failed to produce valid triage output after tool rejection"
        with tracer.start_as_current_span("Final Output") as span7:
            span7.set_attribute("openinference.span.kind", "CHAIN")
            span7.set_attribute("input.value", recovery_output)
            span7.add_event("exception", attributes={
                "exception.type": "AgentExecutionError",
                "exception.message": final_err,
            })
            span7.set_status(Status(StatusCode.ERROR, final_err))

        chain_span.set_attribute("output.value", "Agent failed to produce valid triage output after tool rejection")
        chain_span.set_attribute("metadata.failure_reason", "policy_blocked_unhandled")
        chain_span.set_attribute("latency_ms", round(latency_ms, 2))
        chain_span.set_status(Status(StatusCode.ERROR, "policy_blocked_unhandled"))

In [8]:
# Batch run: 10 traces Scenario A, 10 traces Scenario B (20 total)
NUM_SCENARIO_A = 10
NUM_SCENARIO_B = 10

for i in range(1, NUM_SCENARIO_A + 1):
    case_id = f"AE-2025-048{20 + i}"
    session_id = f"session_ae_a_{i:03d}"
    build_trace_scenario_a(case_id=case_id, session_id=session_id)

for i in range(1, NUM_SCENARIO_B + 1):
    case_id = f"AE-2025-048{30 + i}"
    session_id = f"session_ae_b_{i:03d}"
    build_trace_scenario_b(case_id=case_id, session_id=session_id)

flush_timeout = 15000
flush_success = trace.get_tracer_provider().force_flush(timeout_millis=flush_timeout)
print(f"Flush completed: {flush_success}. Sent {NUM_SCENARIO_A + NUM_SCENARIO_B} traces (10 Scenario A, 10 Scenario B).")

Flush completed: True. Sent 20 traces (10 Scenario A, 10 Scenario B).


## Demo readiness checklist (Step 4)

Verify in Arize AX (or Phoenix):

- **Two agents:** Each trace has **Report Intake Agent** (parent of AE Report Intake + MedDRA/SOP retrievers) and **Triage Classification Agent** (parent of Classification Reasoning; in Scenario B also Tool and Recovery). Final Output remains a direct child of the root.
- **Scenario A (10 traces, case_id AE-2025-04821 … AE-2025-04830):** Each trace has root **OK** and Spans 1–4 only. Click into Span 2 (MedDRA) and Span 3 (SOP) and confirm `document.id` (and score) are visible per retrieved chunk. Filter traces by `metadata.case_id` to audit by case.
- **Scenario B (10 traces, case_id AE-2025-04831 … AE-2025-04840):** Each trace has root **ERROR** and Spans 1–7. Span 5 (Submit to Safety Database) should show the **exception** event and the **policy.action_blocked** event with who/what/why (rule, enforced_by, case_seriousness, required_action). Spans 5 → 6 → 7 should all show red/error status with the cascade visible.
- **Scenario C prep:** Span 4 (Classification Reasoning) has `llm.prompt_template.template`, `llm.prompt_template.version`, and `llm.prompt_template.variables` visible (Arize [prompt template convention](https://arize.com/docs/ax/observe/tracing/configure/instrumenting-prompt-templates-and-prompt-variables)) for version-comparison and Prompt Playground.